# 🌽☕ AgriChain: MindSpore Disease Diagnosis Model Training
This notebook trains high-accuracy models for **Maize** and **Coffee** disease diagnosis using **MindSpore**. 

### 📋 Instructions:
1. Upload your `dataset.zip` (containing `dataset/maize/...` and `dataset/coffee/...`) to this environment.
2. Run all cells.
3. Download the resulting `.ms` files for your Flutter app.

## 🛠 Setup Environment

In [ ]:
!pip install mindspore==2.3.0
import os
import zipfile
import shutil
import random
import numpy as np
from glob import glob
import mindspore as ms
import mindspore.dataset as ds
import mindspore.dataset.vision as vision
import mindspore.dataset.transforms as transforms
from mindspore import nn, ops, train, Model

ms.set_context(mode=ms.GRAPH_MODE, device_target="CPU") # Change to "GPU" or "Ascend" in Huawei Cloud
print(f"MindSpore Version: {ms.__version__}")

## 📦 Step 1: Prepare Dataset (Unzip & Split)
This cell splits your raw folders into **Train (80%)** and **Val (20%)** for better accuracy verification.

In [ ]:
def prepare_data(zip_path, extract_path='./data_raw', split_path='./datasets'):
    if os.path.exists(split_path): shutil.rmtree(split_path)
    
    # 1. Unzip
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print("✅ Unzipped dataset successfully")
    else:
        print("❌ dataset.zip not found! Please upload it.")
        return
    
    # 2. Split for each crop
    # Handle nested folder structure if zip contains a top-level folder
    root_search = os.path.join(extract_path, 'dataset') if os.path.exists(os.path.join(extract_path, 'dataset')) else extract_path
    
    for crop in ['maize', 'coffee']:
        crop_raw = os.path.join(root_search, crop)
        if not os.path.exists(crop_raw): 
            print(f"⚠️ {crop} data not found in {crop_raw}")
            continue
        
        classes = [d for d in os.listdir(crop_raw) if os.path.isdir(os.path.join(crop_raw, d))]
        print(f"Processing {crop} with classes: {classes}")
        
        for cls in classes:
            images = glob(os.path.join(crop_raw, cls, '*'))
            random.shuffle(images)
            
            split_idx = int(len(images) * 0.8)
            train_images = images[:split_idx]
            val_images = images[split_idx:]
            
            # Create directories
            os.makedirs(os.path.join(split_path, crop, 'train', cls), exist_ok=True)
            os.makedirs(os.path.join(split_path, crop, 'val', cls), exist_ok=True)
            
            # Copy files
            for img in train_images: shutil.copy(img, os.path.join(split_path, crop, 'train', cls))
            for img in val_images: shutil.copy(img, os.path.join(split_path, crop, 'val', cls))
            
        print(f"✅ Finished splitting {crop} data into train/val folders")

prepare_data('dataset.zip')

## 📊 Data Pipeline Factory

In [ ]:
def create_dataset(data_path, batch_size=32, training=True):
    dataset = ds.ImageFolderDataset(data_path, decode=True, shuffle=training)
    
    image_size = (224, 224)
    mean = [0.485 * 255, 0.456 * 255, 0.406 * 255]
    std = [0.229 * 255, 0.224 * 255, 0.225 * 255]

    if training:
        trans = [
            vision.RandomCropDecodeResize(image_size, scale=(0.08, 1.0)),
            vision.RandomHorizontalFlip(prob=0.5),
            vision.RandomColorAdjust(brightness=0.4, contrast=0.4),
            vision.Normalize(mean=mean, std=std),
            vision.HWC2CHW()
        ]
    else:
        trans = [
            vision.Resize(256),
            vision.CenterCrop(image_size),
            vision.Normalize(mean=mean, std=std),
            vision.HWC2CHW()
        ]

    dataset = dataset.map(operations=trans, input_columns="image")
    dataset = dataset.map(operations=transforms.TypeCast(ms.int32), input_columns="label")
    dataset = dataset.batch(batch_size, drop_remainder=True)
    return dataset

## 🏗 MobileNetV2 Architecture

In [ ]:
from mindspore.common.initializer import TruncatedNormal

def build_model(num_classes):
    # Standard MobileNetV2 head for classification
    # In a real environment, you'd use mindspore.hub to load pretrained weights
    from mindspore.vision.classification.models import mobilenet_v2
    network = mobilenet_v2(num_classes=num_classes, pretrained=False)
    return network

## 🚀 Training & Export

In [ ]:
def run_training(name, num_classes, base_path, epochs=30):
    train_path = os.path.join(base_path, 'train')
    val_path = os.path.join(base_path, 'val')
    
    if not os.path.exists(train_path): return
    
    train_ds = create_dataset(train_path, training=True)
    val_ds = create_dataset(val_path, training=False)
    
    net = build_model(num_classes)
    loss = nn.SoftmaxCrossEntropyWithLogits(sparse=True, reduction='mean')
    opt = nn.Adam(net.trainable_params(), learning_rate=0.001)
    
    model = Model(net, loss_fn=loss, optimizer=opt, metrics={'acc': nn.Accuracy()})
    
    print(f"--- Training {name} ---")
    model.train(epochs, train_ds, callbacks=[train.LossMonitor()], dataset_sink_mode=False)
    
    acc = model.eval(val_ds)
    print(f"✅ {name} Accuracy: {acc['acc']:.4f}")
    
    # Export to MindIR
    input_tensor = ms.Tensor(np.ones([1, 3, 224, 224]), ms.float32)
    ms.export(net, input_tensor, file_name=f"{name}_disease", file_format="MINDIR")
    print(f"📦 Exported {name}_disease.mindir")
    
    # Helper for converting to Lite in terminal
    print(f"💡 To convert for mobile, run in terminal: converter_lite --fmk=MINDIR --modelFile={name}_disease.mindir --outputFile={name}_disease --inputShape=\"1,3,224,224\"")

# 🟢 Start
run_training("maize", 6, './datasets/maize')
run_training("coffee", 4, './datasets/coffee')